# SkillStudio: neural model (readiness + match + recommendations)
Run the cells top to bottom. Everything is defined inside this notebook, with no extra files to upload or import.

| cells | what they do |
|---|---|
| 1 | imports + config (set the two paths) |
| 2-4 | the model code: featurizer, network, predictor |
| 5-12 | load data, train, calibrate, evaluate, save the `.pt`, demo |
| 13 | OPTIONAL patch: recalibrate the readiness scale inside an already-saved `.pt` (no retraining) |

**Never used as model inputs (leaks):** `match_score`, `admin_approved`, `student_feedback`, `application_status`, `outcome`.

In [ ]:
import os, glob, math, re, time, json
from typing import Iterable, Optional
import numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F
from scipy.stats import spearmanr
from sklearn.metrics import roc_auc_score, cohen_kappa_score, accuracy_score, f1_score

CFG = dict(
    data_dir="/kaggle/input", out_dir="/kaggle/working/outputs",
    d=64, h=128, dropout=0.15, lr=2e-3, weight_decay=1e-4, batch_size=512, epochs=40, patience=6,
    aux_weight=0.25,                 # weight of the auxiliary application-status head (0 disables it)
    split="student",                 # "student" = grouped by student (no leakage)  |  "time" = last 10% of dates = test
    val_frac=0.10, test_frac=0.10, seed=42,
    readiness_topk=10, reach_cut=0.6,       # reach = share of the job bank where match >= 60%
    readiness_weights=dict(market=0.15, reach=0.35, skills=0.25, evidence=0.25), readiness_ref_students=4000)
torch.manual_seed(CFG["seed"]); np.random.seed(CFG["seed"])
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu"); print("device:", DEVICE)

## Model code (cells 2-4): just run them, nothing to edit

In [ ]:
# ----------------------------------------------------------------------------- constants
PROF = {"beginner": 0.4, "intermediate": 0.7, "advanced": 1.0, "expert": 1.0}
FB_MAP = {"Not Relevant": 0, "Somewhat Relevant": 1, "Relevant": 2}
ST_MAP = {"Not Applied": 0, "Applied": 1, "Shortlisted": 2, "Interviewed": 3, "Offered": 4}
ROLE_NOISE = {"junior", "senior", "intern", "lead", "associate", "trainee", "graduate", "entry", "level", "internship"}
PAIR_FEATURES = ["req_cov_exact", "req_cov_credit", "req_cov_strength", "req_cov_advanced", "req_hits", "req_missing",
                 "pref_cov_exact", "pref_cov_credit", "proj_cov", "role_match", "degree_ok", "exp_gap", "exp_ok",
                 "mode_ok", "type_ok", "loc_ok"]
F_IDX = {n: i for i, n in enumerate(PAIR_FEATURES)}
N_PF = len(PAIR_FEATURES)


# ----------------------------------------------------------------------------- parsing helpers
def _null(v) -> bool:
    return v is None or (isinstance(v, float) and math.isnan(v)) or (not isinstance(v, (str, list)) and pd.isna(v))


def split_pipe(v) -> list[str]:
    return [] if _null(v) else [t.strip() for t in str(v).split("|") if t.strip()]


def norm(x) -> str:
    return re.sub(r"[^a-z0-9]", "", str(x).lower())


def parse_proficiency(v) -> dict[str, float]:
    out = {}
    for part in split_pipe(v):
        if ":" in part:
            name, lvl = part.rsplit(":", 1)
            out[name.strip().lower()] = PROF.get(lvl.strip().lower(), 0.5)
    return out


def core_role(title) -> str:
    return " ".join(w for w in re.findall(r"[a-z0-9+#./]+", str(title).lower()) if w not in ROLE_NOISE)


def level_index(title) -> int:
    w = str(title).lower().split()
    return 0 if "intern" in w else 1 if "junior" in w else 3 if ("senior" in w or "lead" in w) else 2


def _num(r, k, default=0.0) -> float:
    v = r.get(k, default)
    try:
        v = float(v)
        return default if math.isnan(v) else v
    except (TypeError, ValueError):
        return default


def _onehot(idx: dict, key, n) -> np.ndarray:
    v = np.zeros(n, dtype=np.float32)
    if key in idx:
        v[idx[key]] = 1.0
    return v


# ----------------------------------------------------------------------------- featurizer
class Featurizer:
    """Turns raw records (dicts with the CSV column names) into model tensors. Vocab is stored in the .pt."""

    def fit(self, students: pd.DataFrame, jobs: pd.DataFrame) -> "Featurizer":
        S, J = students.to_dict("records"), jobs.to_dict("records")
        skills = set()
        for r in S:
            skills |= set(self._student_skill_names(r))
        for r in J:
            skills |= {s.lower() for s in split_pipe(r["required_skills"]) + split_pipe(r["preferred_skills"])}
        degs = {norm(r["degree"]) for r in S} | {norm(d) for r in J for d in split_pipe(r["required_degree"])}
        degs.discard(norm("Any Degree")); degs.discard("")
        self.skills = sorted(skills)
        self.roles = sorted({core_role(r["job_title"]) for r in J} | {x.lower() for r in S for x in split_pipe(r.get("preferred_roles"))})
        self.degrees = sorted(degs)
        self.specs = sorted({norm(r["specialization"]) for r in S})
        self.job_types = sorted({norm(r["job_type"]) for r in J})
        self.job_modes = sorted({norm(r["work_mode"]) for r in J})
        self.locations = sorted({norm(r["location"]) for r in J} | {norm(x) for r in S for x in split_pipe(r.get("preferred_locations"))})
        self._build_index()
        return self

    def _build_index(self):
        self.sk_i = {s: i for i, s in enumerate(self.skills)}
        self.ro_i = {s: i for i, s in enumerate(self.roles)}
        self.dg_i = {s: i for i, s in enumerate(self.degrees)}
        self.sp_i = {s: i for i, s in enumerate(self.specs)}
        self.jt_i = {s: i for i, s in enumerate(self.job_types)}
        self.jm_i = {s: i for i, s in enumerate(self.job_modes)}
        self.lo_i = {s: i for i, s in enumerate(self.locations)}
        alt = "|".join(sorted(map(re.escape, self.skills), key=len, reverse=True))
        self._cert_re = re.compile(r"(?<![a-z0-9+#])(" + alt + r")(?![a-z0-9+#])") if alt else None

    def state(self) -> dict:
        return {k: list(getattr(self, k)) for k in ("skills", "roles", "degrees", "specs", "job_types", "job_modes", "locations")}

    @classmethod
    def from_state(cls, st: dict) -> "Featurizer":
        fz = cls()
        for k, v in st.items():
            setattr(fz, k, list(v))
        fz._build_index()
        return fz

    @property
    def dims(self) -> dict:
        K, R, Dg, Sp, T, M, Lc = (len(x) for x in (self.skills, self.roles, self.degrees, self.specs, self.job_types, self.job_modes, self.locations))
        return dict(K=K, R=R, s_ctx=7 + Dg + Sp + M + T + Lc, j_ctx=2 + 4 + T + M + Lc + Dg + 1)

    # ---- skills
    def _student_skill_names(self, r) -> list[str]:
        names = [s.lower() for k in ("technical_skills", "programming_languages", "project_technologies") for s in split_pipe(r.get(k))]
        return names + list(parse_proficiency(r.get("skill_proficiency")))

    def unrecognized_skills(self, r) -> list[str]:
        return sorted({s for s in self._student_skill_names(r) if s not in self.sk_i})

    def _strength(self, r) -> tuple[np.ndarray, np.ndarray]:
        """Per-skill evidence strength in [0,1] (proficiency + project usage + certification mention) and project mask."""
        v = np.zeros(len(self.skills), dtype=np.float32)
        pm = np.zeros(len(self.skills), dtype=np.float32)
        prof = parse_proficiency(r.get("skill_proficiency"))
        tech = {s.lower() for s in split_pipe(r.get("technical_skills"))}
        langs = {s.lower() for s in split_pipe(r.get("programming_languages"))}
        proj = {s.lower() for s in split_pipe(r.get("project_technologies"))}
        certs = set()
        if not _null(r.get("certifications")) and self._cert_re is not None:
            certs = set(self._cert_re.findall(str(r["certifications"]).lower()))
        for s in tech | langs | proj | set(prof):
            i = self.sk_i.get(s)
            if i is None:
                continue
            base = prof.get(s)
            if base is None:
                base = 0.6 if s in langs else 0.5 if s in tech else 0.35     # project-only = weakest evidence
            if s in proj and (s in tech or s in langs or s in prof):
                base += 0.15
            if s in certs:
                base += 0.10
            v[i] = min(1.0, base)
            pm[i] = 1.0 if s in proj else 0.0
        return v, pm

    # ---- students
    def encode_students(self, records: Iterable[dict]) -> dict[str, torch.Tensor]:
        rows = {k: [] for k in ("strength", "proj", "role", "ctx", "deg", "mode", "type", "loc", "exp", "stats")}
        nM, nT, nL = len(self.job_modes), len(self.job_types), len(self.locations)
        for r in records:
            sv, pm = self._strength(r)
            roles = np.zeros(len(self.roles), dtype=np.float32)
            for x in split_pipe(r.get("preferred_roles")):
                if x.lower() in self.ro_i:
                    roles[self.ro_i[x.lower()]] = 1.0
            projects, intern, work = _num(r, "projects_count"), _num(r, "internship_months"), _num(r, "work_experience_months")
            certs = len(split_pipe(r.get("certifications")))
            deg, spec = _onehot(self.dg_i, norm(r.get("degree")), len(self.degrees)), _onehot(self.sp_i, norm(r.get("specialization")), len(self.specs))
            pmode = norm(r.get("preferred_work_mode"))
            mode = np.ones(nM, dtype=np.float32) if pmode in ("flexible", "any", "") else _onehot(self.jm_i, pmode, nM)
            ptype = norm(r.get("job_type_preference"))
            jtyp = np.ones(nT, dtype=np.float32) if ptype in ("any", "flexible", "") else _onehot(self.jt_i, ptype, nT)
            loc = np.zeros(nL, dtype=np.float32)
            for x in split_pipe(r.get("preferred_locations")):
                if norm(x) in self.lo_i:
                    loc[self.lo_i[norm(x)]] = 1.0
            if "remote" in self.lo_i:
                loc[self.lo_i["remote"]] = 1.0          # remote jobs suit everyone
            if loc.sum() <= 1 and not split_pipe(r.get("preferred_locations")):
                loc[:] = 1.0                            # no stated preference = anywhere
            total = intern + work
            dense = np.array([projects / 7, intern / 12, work / 28, certs / 4, (_num(r, "graduation_year", 2026) - 2026) / 2,
                              total / 40, (sv > 0).sum() / 20], dtype=np.float32)
            rows["strength"].append(sv); rows["proj"].append(pm); rows["role"].append(roles)
            rows["ctx"].append(np.concatenate([dense, deg, spec, mode, jtyp, loc]))
            rows["deg"].append(deg); rows["mode"].append(mode); rows["type"].append(jtyp); rows["loc"].append(loc)
            rows["exp"].append(np.float32(total)); rows["stats"].append(np.array([projects, intern, work, certs], dtype=np.float32))
        return {k: torch.from_numpy(np.stack(v)) for k, v in rows.items()}

    # ---- jobs
    def encode_jobs(self, records: Iterable[dict]) -> dict[str, torch.Tensor]:
        rows = {k: [] for k in ("req", "pref", "role", "ctx", "req_deg", "any_deg", "jmode", "jtype", "jloc", "min_exp")}
        K, T, M, Lc, Dg = len(self.skills), len(self.job_types), len(self.job_modes), len(self.locations), len(self.degrees)
        for r in records:
            req, pref = np.zeros(K, dtype=np.float32), np.zeros(K, dtype=np.float32)
            for s in split_pipe(r["required_skills"]):
                if s.lower() in self.sk_i: req[self.sk_i[s.lower()]] = 1.0
            for s in split_pipe(r["preferred_skills"]):
                if s.lower() in self.sk_i: pref[self.sk_i[s.lower()]] = 1.0
            role = _onehot(self.ro_i, core_role(r["job_title"]), len(self.roles))
            lvl = np.zeros(4, dtype=np.float32); lvl[level_index(r["job_title"])] = 1.0
            jt, jm, jl = _onehot(self.jt_i, norm(r["job_type"]), T), _onehot(self.jm_i, norm(r["work_mode"]), M), _onehot(self.lo_i, norm(r["location"]), Lc)
            degs = [norm(d) for d in split_pipe(r.get("required_degree"))]
            any_deg = np.array([1.0 if (not degs or norm("Any Degree") in degs) else 0.0], dtype=np.float32)
            rd = np.zeros(Dg, dtype=np.float32)
            for d in degs:
                if d in self.dg_i: rd[self.dg_i[d]] = 1.0
            mn, mx = _num(r, "min_experience_months"), _num(r, "max_experience_months")
            ctx = np.concatenate([np.array([mn / 24, mx / 60], dtype=np.float32), lvl, jt, jm, jl, rd, any_deg])
            for k, v in (("req", req), ("pref", pref), ("role", role), ("ctx", ctx), ("req_deg", rd), ("any_deg", any_deg),
                         ("jmode", jm), ("jtype", jt), ("jloc", jl), ("min_exp", np.float32(mn))):
                rows[k].append(v)
        return {k: torch.from_numpy(np.stack(v)) for k, v in rows.items()}


def skill_relations(S: dict, J: dict, cap=0.5, min_sim=0.45, dim=32) -> torch.Tensor:
    """Data-driven skill relatedness (PPMI of co-occurrence -> SVD -> cosine). S[a,b] = credit for having a when b is required."""
    docs = torch.cat([((J["req"] + J["pref"]) > 0).float(), (S["strength"] > 0).float()]).numpy()
    C = docs.T @ docs
    np.fill_diagonal(C, 0)
    tot, rs = C.sum(), C.sum(1, keepdims=True)
    with np.errstate(divide="ignore", invalid="ignore"):
        pmi = np.where(C > 0, np.log(C * tot / (rs @ rs.T)), 0.0)
    pmi = np.maximum(np.nan_to_num(pmi), 0)
    U, s, _ = np.linalg.svd(pmi, full_matrices=False)
    d = min(dim, len(s) - 1)
    emb = U[:, :d] * np.sqrt(s[:d])
    emb /= np.maximum(np.linalg.norm(emb, axis=1, keepdims=True), 1e-9)
    sim = np.clip(emb @ emb.T, 0, 1)
    out = np.where(sim >= min_sim, sim * cap, 0.0)
    np.fill_diagonal(out, 1.0)
    return torch.tensor(out, dtype=torch.float32)

In [ ]:
# ----------------------------------------------------------------------------- network
def _mlp(i, h, o, p):
    return nn.Sequential(nn.Linear(i, h), nn.GELU(), nn.Dropout(p), nn.Linear(h, o), nn.LayerNorm(o))


def _descending(base: torch.Tensor, raw: torch.Tensor) -> list[torch.Tensor]:
    """thresholds b_1 >= b_2 >= ... >= b_n  (base is the smallest)."""
    out, cur = [base], base
    for r in raw:
        cur = cur + F.softplus(r)
        out.append(cur)
    return out[::-1]


class SkillStudioNet(nn.Module):
    """
    Two towers (student / job) over a shared skill-embedding table  +  explicit differentiable pair features
    (coverage, missing skills, role match, ...)  ->  wide&deep head with two ORDINAL (CORAL-style) outputs:
        z      latent fit            -> P(feedback >= Somewhat), P(feedback >= Relevant)   (match score)
        z_aux  latent success        -> P(status >= Applied/Shortlisted/Interviewed/Offered)
    """

    def __init__(self, dims: dict, S_rel: torch.Tensor, d=64, h=128, p=0.15):
        super().__init__()
        self.dims, self.d, self.h, self.p = dims, d, h, p
        K, R = dims["K"], dims["R"]
        self.skill_emb = nn.Parameter(torch.randn(K, d) * 0.1)
        self.role_emb = nn.Parameter(torch.randn(R, d) * 0.1)
        self.register_buffer("S_rel", S_rel.clone())
        self.rel_gate = nn.Parameter(torch.tensor(0.0))                 # sigmoid(0)=0.5 weight on related-skill credit
        self.s_tower = _mlp(3 * d + dims["s_ctx"], h, d, p)
        self.j_tower = _mlp(3 * d + dims["j_ctx"], h, d, p)
        self.deep = nn.Sequential(nn.Linear(3 * d + N_PF, h), nn.GELU(), nn.Dropout(p), nn.Linear(h, h // 2), nn.GELU())
        self.wide = nn.Linear(N_PF, 1)
        self.out_rel, self.out_aux = nn.Linear(h // 2, 1), nn.Linear(h // 2, 1)
        self.rel_scale = nn.Parameter(torch.tensor(1.0))                 # slope/offsets also fitted post-hoc on validation (calibration)
        self.rel_b2, self.rel_raw = nn.Parameter(torch.tensor(-0.5)), nn.Parameter(torch.zeros(1))
        self.st_b4, self.st_raw = nn.Parameter(torch.tensor(-1.0)), nn.Parameter(torch.zeros(3))

    def config(self) -> dict:
        return dict(dims=self.dims, d=self.d, h=self.h, p=self.p)

    # --- encoders
    @staticmethod
    def _bag(w, E):
        return (w @ E) / (w.sum(-1, keepdim=True) + 1e-6)

    def encode_student(self, S):
        return self.s_tower(torch.cat([self._bag(S["strength"], self.skill_emb), self._bag(S["proj"], self.skill_emb),
                                       self._bag(S["role"], self.role_emb), S["ctx"]], -1))

    def encode_job(self, J):
        return self.j_tower(torch.cat([self._bag(J["req"], self.skill_emb), self._bag(J["pref"], self.skill_emb),
                                       J["role"] @ self.role_emb, J["ctx"]], -1))

    def credit(self, strength):
        """Soft skill credit: own strength, or strength x relation to a related skill (<= 0.5), gated by a learned scalar."""
        rel = (strength.unsqueeze(-1) * self.S_rel.unsqueeze(0)).amax(1)
        return torch.maximum(strength, torch.sigmoid(self.rel_gate) * rel)

    # --- explicit pair features; paired=True -> row-wise (B,), paired=False -> all combinations (B,N)
    def pair_feats(self, S, J, paired):
        dot = (lambda a, b: (a * b).sum(-1)) if paired else (lambda a, b: a @ b.T)
        ex = (lambda t: t) if paired else (lambda t: t.unsqueeze(0))        # job-side vectors -> broadcast over students
        rs_raw, ps = J["req"].sum(-1), J["pref"].sum(-1)
        rs, ps = ex(rs_raw.clamp(min=1)), ex(ps.clamp(min=1))
        sx, adv = (S["strength"] > 0).float(), (S["strength"] >= 0.95).float()
        cr = self.credit(S["strength"])
        hits = dot(sx, J["req"])
        exp = S["exp"] if paired else S["exp"].unsqueeze(1)
        gap = ((ex(J["min_exp"]) - exp) / 24).clamp(-1, 1)
        deg = (dot(S["deg"], J["req_deg"]) + ex(J["any_deg"].squeeze(-1))).clamp(max=1)
        f = [hits / rs, dot(cr, J["req"]) / rs, dot(S["strength"], J["req"]) / rs, dot(adv, J["req"]) / rs, hits / 7, (rs - hits) / 7,
             dot(sx, J["pref"]) / ps, dot(cr, J["pref"]) / ps, dot(S["proj"], J["req"]) / rs, dot(S["role"], J["role"]), deg, gap,
             (gap <= 0).float(), dot(S["mode"], J["jmode"]).clamp(max=1), dot(S["type"], J["jtype"]).clamp(max=1), dot(S["loc"], J["jloc"]).clamp(max=1)]
        return torch.stack(f, -1)

    def forward(self, S, J, paired=True):
        se, je = self.encode_student(S), self.encode_job(J)
        pf = self.pair_feats(S, J, paired)
        if paired:
            x = torch.cat([se, je, se * je, pf], -1)
        else:
            B, N = se.shape[0], je.shape[0]
            x = torch.cat([se[:, None].expand(B, N, -1), je[None].expand(B, -1, -1), se[:, None] * je[None], pf], -1)
        hd = self.deep(x)
        return self.out_rel(hd).squeeze(-1) + self.wide(pf).squeeze(-1), self.out_aux(hd).squeeze(-1), pf

    # --- ordinal heads
    def rel_logits(self, z):
        b1 = self.rel_b2 + F.softplus(self.rel_raw[0])
        zs = z * self.rel_scale
        return zs + b1, zs + self.rel_b2                                 # logits of P(y>=1), P(y>=2)

    def status_logits(self, z_aux):
        b = _descending(self.st_b4, self.st_raw)                         # 4 thresholds, descending
        return torch.stack([z_aux + t for t in b], -1)

    def match_outputs(self, z):
        l1, l2 = self.rel_logits(z)
        p1, p2 = torch.sigmoid(l1), torch.sigmoid(l2)
        return (p1 + p2) / 2, torch.stack([1 - p1, p1 - p2, p2], -1)     # expected relevance in [0,1], class probabilities

In [ ]:
# ----------------------------------------------------------------------------- checkpoint IO
META_COLS = ["job_id", "job_title", "company", "location", "work_mode", "job_type", "min_experience_months",
             "max_experience_months", "required_degree", "job_description"]


def job_meta(jobs_df: pd.DataFrame) -> dict:
    conv = lambda v: None if _null(v) else (float(v) if isinstance(v, (int, float, np.integer, np.floating)) else str(v))
    return {c: [conv(v) for v in jobs_df[c]] for c in META_COLS if c in jobs_df}


def build_checkpoint(net, fz, jobs_df, job_tensors, readiness_ref, cfg, metrics=None) -> dict:
    return dict(format_version=1, net_config=net.config(), state_dict={k: v.detach().cpu() for k, v in net.state_dict().items()},
                featurizer=fz.state(), job_bank=dict(tensors={k: v.cpu() for k, v in job_tensors.items()}, meta=job_meta(jobs_df)),
                readiness_ref=torch.as_tensor(readiness_ref, dtype=torch.float32).cpu(), cfg=cfg, metrics=metrics or {})


def save_checkpoint(path, ckpt: dict):
    torch.save(ckpt, path)      # only tensors / lists / dicts / str / numbers -> loads with weights_only=True
    return path


# ----------------------------------------------------------------------------- predictor
READINESS_KEYS = ["market", "reach", "skills", "evidence"]


def _denom(lo: float, hi: float) -> float:
    """Width of the p5..p95 scaling window; floored at 5% of the component's own upper reference so that
    saturated components do not explode, without squashing naturally small-valued ones (e.g. reach ~ 0.01-0.04)."""
    return max(hi - lo, 0.05 * abs(hi), 1e-6)


class SkillStudioPredictor:
    """Loads the .pt and serves readiness / match / recommendation outputs."""

    def __init__(self, ckpt: dict, device: Optional[str] = None):
        self.device = torch.device(device or ("cuda" if torch.cuda.is_available() else "cpu"))
        self.cfg = ckpt["cfg"]
        self.fz = Featurizer.from_state(ckpt["featurizer"])
        c = ckpt["net_config"]
        self.net = SkillStudioNet(c["dims"], torch.zeros(c["dims"]["K"], c["dims"]["K"]), c["d"], c["h"], c["p"])
        res = self.net.load_state_dict(ckpt["state_dict"], strict=False)       # older checkpoints may lack rel_scale (defaults to 1.0)
        if res.unexpected_keys or set(res.missing_keys) - {"rel_scale"}:
            raise RuntimeError(f"checkpoint/architecture mismatch: missing={res.missing_keys} unexpected={res.unexpected_keys}")
        self.net.to(self.device).eval()
        self.ref = ckpt["readiness_ref"].to(self.device)
        self.scale = ckpt.get("readiness_scale") or {k: [0.0, 1.0] for k in READINESS_KEYS}
        self.metrics = ckpt.get("metrics", {})
        self._set_bank(ckpt["job_bank"]["tensors"], ckpt["job_bank"]["meta"])

    @classmethod
    def load(cls, path, device=None):
        return cls(torch.load(path, map_location="cpu", weights_only=True), device)

    # ---- job bank
    def _set_bank(self, tensors, meta):
        self.J = {k: v.to(self.device) for k, v in tensors.items()}
        self.meta = meta
        self.n_jobs = self.J["req"].shape[0]
        self.job_role = self.J["role"].argmax(-1)
        self.job_index = {j: i for i, j in enumerate(meta["job_id"])}

    def set_job_bank(self, jobs_df: pd.DataFrame):
        """Swap in a fresh job list (new postings) without retraining."""
        self._set_bank(self.fz.encode_jobs(jobs_df.to_dict("records")), job_meta(jobs_df))

    # ---- core scoring
    @torch.no_grad()
    def match_matrix(self, S: dict, job_idx=None, return_pf=False, max_rows=120_000):
        S = {k: v.to(self.device) for k, v in S.items()}
        J = self.J if job_idx is None else {k: v[job_idx] for k, v in self.J.items()}
        B, N = S["strength"].shape[0], J["req"].shape[0]
        step = max(1, max_rows // max(N, 1))
        match, probs, offer, pfs = [], [], [], []
        for a in range(0, B, step):
            Sb = {k: v[a:a + step] for k, v in S.items()}
            z, za, pf = self.net(Sb, J, paired=False)
            m, pr = self.net.match_outputs(z)
            match.append(m); probs.append(pr)
            offer.append(torch.sigmoid(self.net.status_logits(za)[..., 3]))
            if return_pf: pfs.append(pf)
        out = dict(match=torch.cat(match), probs=torch.cat(probs), offer=torch.cat(offer))
        if return_pf: out["pf"] = torch.cat(pfs)
        return out

    # ---- readiness: 4 components, each robust-scaled against the cohort reference (so scores spread out instead of saturating)
    def _components(self, S, out) -> dict:
        k, cut = self.cfg["readiness_topk"], self.cfg.get("reach_cut", 0.6)
        elig = out["pf"][..., F_IDX["degree_ok"]] > 0.5
        m = torch.where(elig, out["match"], torch.zeros_like(out["match"]))
        market = m.topk(min(k, m.shape[1]), dim=1).values.mean(1)            # quality of the best matches
        reach = (m >= cut).float().sum(1) / m.shape[1]                       # share of the job market they are a good match for
        sv = S["strength"].to(self.device)
        top8 = sv.topk(min(8, sv.shape[1]), dim=1).values
        nz = (top8 > 0).sum(1)
        depth = torch.where(nz > 0, top8.sum(1) / nz.clamp(min=1), torch.zeros_like(market))
        skills = 0.6 * depth + 0.4 * ((sv > 0).sum(1) / 12).clamp(max=1)     # skill depth + breadth
        st = S["stats"].to(self.device)
        evidence = (0.35 * (st[:, 0] / 4).clamp(max=1) + 0.30 * (st[:, 1] / 9).clamp(max=1) + 0.20 * (st[:, 2] / 24).clamp(max=1)
                    + 0.15 * (st[:, 3] / 2).clamp(max=1))
        return dict(market=market, reach=reach, skills=skills, evidence=evidence, masked_match=m, elig=elig)

    def _scale(self, key, x):
        lo, hi = self.scale[key]
        return ((x - lo) / _denom(lo, hi)).clamp(0, 1)

    def _readiness_from(self, S, out) -> dict:
        c = self._components(S, out)
        w = self.cfg["readiness_weights"]
        scaled = {k: self._scale(k, c[k]) for k in READINESS_KEYS}
        c["scaled"], c["raw"] = scaled, sum(w[k] * scaled[k] for k in READINESS_KEYS)
        return c

    @torch.no_grad()
    def _component_table(self, students: pd.DataFrame, batch=64) -> pd.DataFrame:
        recs, rows = students.to_dict("records"), []
        for a in range(0, len(recs), batch):
            S = self.fz.encode_students(recs[a:a + batch])
            c = self._components(S, self.match_matrix(S, return_pf=True))
            for i, r in enumerate(recs[a:a + batch]):
                rows.append(dict(student_id=r.get("student_id"), **{k: float(c[k][i]) for k in READINESS_KEYS}))
        return pd.DataFrame(rows)

    def _finish(self, df: pd.DataFrame) -> pd.DataFrame:
        w = self.cfg["readiness_weights"]
        for k in READINESS_KEYS:
            lo, hi = self.scale[k]; df[k + "_scaled"] = ((df[k] - lo) / _denom(lo, hi)).clip(0, 1)
        df["readiness_raw"] = sum(w[k] * df[k + "_scaled"] for k in READINESS_KEYS)
        df["readiness_score"] = (df.readiness_raw * 100).round(1)
        df["percentile"] = (np.searchsorted(np.sort(self.ref.cpu().numpy()), df.readiness_raw) / max(len(self.ref), 1) * 100).round(1)
        return df

    def readiness_table(self, students: pd.DataFrame, batch=64) -> pd.DataFrame:
        return self._finish(self._component_table(students, batch))

    def calibrate_readiness(self, sample: pd.DataFrame, lo_q=5, hi_q=95) -> pd.DataFrame:
        """Fit the cohort reference on a student sample: per-component p5..p95 scaling + the score distribution for percentiles."""
        self.scale = {k: [0.0, 1.0] for k in READINESS_KEYS}
        df = self._component_table(sample)
        self.scale = {k: [float(np.percentile(df[k], lo_q)), float(np.percentile(df[k], hi_q))] for k in READINESS_KEYS}
        w = self.cfg["readiness_weights"]
        raw = sum(w[k] * ((df[k] - self.scale[k][0]) / _denom(*self.scale[k])).clip(0, 1) for k in READINESS_KEYS)
        self.ref = torch.tensor(np.sort(raw.values), dtype=torch.float32, device=self.device)
        return self._finish(df)

    def export_calibration(self) -> dict:
        return dict(readiness_scale={k: list(v) for k, v in self.scale.items()}, readiness_ref=self.ref.cpu())

    def _role_readiness(self, masked_match):
        res = []
        for r in range(len(self.fz.roles)):
            cols = (self.job_role == r).nonzero().squeeze(-1)
            res.append(masked_match[:, cols].topk(min(5, len(cols)), dim=1).values.mean(1) if len(cols) else torch.zeros(masked_match.shape[0], device=self.device))
        return torch.stack(res, 1)

    # ---- public API
    def _records(self, student):
        if isinstance(student, pd.DataFrame): return student.to_dict("records")
        return [student] if isinstance(student, dict) else list(student)

    def fit_label(self, match_pct: float, req_coverage: float = 1.0) -> str:
        c1, c2 = self.cfg.get("fit_cutoffs", [40.0, 70.0])
        if req_coverage < self.cfg.get("min_required_coverage", 0.25):
            return "Stretch"                # UX guard: almost none of the required skills are covered
        if match_pct >= c2 and req_coverage < self.cfg.get("strong_min_required_coverage", 0.5):
            return "Partial match"          # UX guard: never call it "Strong" when under half of the required skills are covered
        return "Strong match" if match_pct >= c2 else "Partial match" if match_pct >= c1 else "Stretch"

    @torch.no_grad()
    def analyze(self, student, top_n=10, pref_weight=0.15, max_per_role=3, only_eligible=True, job_type=None) -> dict:
        rec = self._records(student)[0]
        S = self.fz.encode_students([rec])
        out = self.match_matrix(S, return_pf=True)
        rd = self._readiness_from(S, out)
        match, pf = out["match"][0], out["pf"][0]
        elig = rd["elig"][0] if only_eligible else torch.ones_like(match, dtype=torch.bool)
        if job_type and self.fz.jt_i.get(norm(job_type)) is not None:
            elig = elig & (self.J["jtype"][:, self.fz.jt_i[norm(job_type)]] > 0.5)
        interest = 0.4 * pf[:, F_IDX["mode_ok"]] + 0.4 * pf[:, F_IDX["type_ok"]] + 0.2 * pf[:, F_IDX["loc_ok"]]
        rank = torch.where(elig, match * (1 - pref_weight + pref_weight * interest), torch.full_like(match, -1.0))
        picks, per_role = [], {}
        for j in rank.argsort(descending=True).tolist():
            if rank[j] < 0 or len(picks) >= top_n: break
            r = int(self.job_role[j])
            if max_per_role and per_role.get(r, 0) >= max_per_role: continue
            per_role[r] = per_role.get(r, 0) + 1
            picks.append(j)

        sv = S["strength"][0].to(self.device)
        credit = self.net.credit(sv.unsqueeze(0))[0]
        recs = [self._explain(j, out, sv, credit) for j in picks]
        pct = float((self.ref < rd["raw"][0]).float().mean() * 100)
        band = "Strong" if pct >= 85 else "Competitive" if pct >= 55 else "Developing" if pct >= 25 else "Emerging"
        sc = rd["scaled"]
        comps = dict(match_quality=round(float(sc["market"][0]) * 100), opportunity_reach=round(float(sc["reach"][0]) * 100),
                     skill_profile=round(float(sc["skills"][0]) * 100), experience_evidence=round(float(sc["evidence"][0]) * 100))
        n_good = int((rd["masked_match"][0] >= self.cfg.get("reach_cut", 0.6)).sum())
        rr = self._role_readiness(rd["masked_match"])[0].cpu().numpy()
        role_ready = [dict(role=self.fz.roles[i], readiness=round(float(rr[i]) * 100, 1)) for i in np.argsort(-rr)[:5]]
        roi = self._skill_roi(S, rd, out)
        advice = []
        if roi: advice.append(f"Learning '{roi[0]['skill']}' would add about {roi[0]['readiness_gain_points']} readiness points "
                              f"and open {roi[0]['new_good_matches']} more good matches.")
        if comps["experience_evidence"] < 50: advice.append("Add a project or internship; practical evidence is a weak spot.")
        if comps["skill_profile"] < 50: advice.append("Deepen 3-4 core skills to Intermediate/Advanced rather than adding more skills.")
        return dict(readiness=dict(score=round(float(rd["raw"][0]) * 100, 1), percentile=round(pct, 1), band=band, components=comps,
                                   good_matches=n_good, eligible_jobs=int(rd["elig"][0].sum()), role_readiness=role_ready, advice=advice),
                    recommendations=recs, skill_roi=roi, unrecognized_skills=self.fz.unrecognized_skills(rec))

    def _explain(self, j, out, sv, credit) -> dict:
        names = self.fz.skills
        req = (self.J["req"][j] > 0).nonzero().squeeze(-1).tolist()
        pref = (self.J["pref"][j] > 0).nonzero().squeeze(-1).tolist()
        lvl = lambda s: "Advanced" if s >= 0.95 else "Intermediate" if s >= 0.65 else "Beginner"
        matched = [dict(skill=names[s], level=lvl(float(sv[s]))) for s in req if sv[s] > 0]
        related = [dict(required=names[s], via=names[int((sv * self.net.S_rel[:, s]).argmax())]) for s in req if sv[s] == 0 and credit[s] >= 0.3]
        missing = [names[s] for s in req if sv[s] == 0 and credit[s] < 0.3]
        m = float(out["match"][0, j]) * 100
        pr = out["probs"][0, j].tolist()
        get = lambda k: self.meta[k][j] if k in self.meta else None
        return dict(job_id=get("job_id"), job_title=get("job_title"), company=get("company"), location=get("location"),
                    work_mode=get("work_mode"), job_type=get("job_type"), match_score=round(m, 1), fit_label=self.fit_label(m, (len(matched) + len(related)) / max(len(req), 1)),
                    probabilities=dict(not_relevant=round(pr[0], 3), somewhat=round(pr[1], 3), relevant=round(pr[2], 3)),
                    offer_likelihood=round(float(out["offer"][0, j]), 3), required_skills_met=f"{len(matched)}/{len(req)}",
                    matched_skills=matched, related_skills=related, missing_required=missing,
                    missing_preferred=[names[s] for s in pref if sv[s] == 0 and credit[s] < 0.3],
                    min_experience_months=get("min_experience_months"), job_description=get("job_description"))

    @torch.no_grad()
    def match_jobs(self, student, job_ids: list[str]) -> list[dict]:
        """Match score of one student against specific jobs."""
        S = self.fz.encode_students(self._records(student)[:1])
        out = self.match_matrix(S, return_pf=True)
        sv = S["strength"][0].to(self.device)
        credit = self.net.credit(sv.unsqueeze(0))[0]
        return [self._explain(self.job_index[j], out, sv, credit) for j in job_ids if j in self.job_index]

    @torch.no_grad()
    def _skill_roi(self, S, rd, out, n_cand=20, top=5):
        """Counterfactual: add a missing skill at Intermediate level, re-score the whole job bank, measure the readiness gain."""
        sv = S["strength"][0].to(self.device)
        top_jobs = out["match"][0].masked_fill(~rd["elig"][0], -1).topk(min(30, self.n_jobs)).indices
        need = (self.J["req"][top_jobs] + 0.5 * self.J["pref"][top_jobs]).sum(0) * (sv == 0).float()
        cand = need.topk(min(n_cand, int((need > 0).sum()))).indices
        if len(cand) == 0: return []
        Sv = {k: v.to(self.device).expand(len(cand), *v.shape[1:]).clone() for k, v in S.items()}
        Sv["strength"][torch.arange(len(cand)), cand] = 0.7
        r2 = self._readiness_from(Sv, self.match_matrix(Sv, return_pf=True))
        good0 = int((rd["masked_match"][0] >= self.cfg.get("reach_cut", 0.6)).sum())
        res = []
        for i, k in enumerate(cand.tolist()):
            res.append(dict(skill=self.fz.skills[k], readiness_gain_points=round(float(r2["raw"][i] - rd["raw"][0]) * 100, 2),
                            new_good_matches=int((r2["masked_match"][i] >= self.cfg.get("reach_cut", 0.6)).sum()) - good0))
        return sorted(res, key=lambda d: (-d["readiness_gain_points"], -d["new_good_matches"]))[:top]

## Load data and featurize

In [ ]:
# ---- 1. load (files are found by name) ----
files = glob.glob(f"{CFG['data_dir']}/**/*.csv", recursive=True)
def pick(keys, ex=()): return next(f for f in files if f not in ex and any(k in os.path.basename(f).lower() for k in keys))
f_fb = pick(["feedback", "recommend"]); f_st = pick(["student"], [f_fb]); f_jb = pick(["job"], [f_fb, f_st])
students, jobs, fb = pd.read_csv(f_st), pd.read_csv(f_jb), pd.read_csv(f_fb)

# ---- 2. targets only (leak columns stay out of the model) ----
fb = fb[fb.student_id.isin(students.student_id) & fb.job_id.isin(jobs.job_id)].copy()
fb["y"] = fb.student_feedback.map(FB_MAP)                         # 0 Not / 1 Somewhat / 2 Relevant
fb["st"] = fb.application_status.map(ST_MAP).fillna(-1)           # 0..4, -1 = unknown (masked in aux loss)
fb = fb.dropna(subset=["y"]).reset_index(drop=True); fb["y"] = fb.y.astype(int)
s_idx = {s: i for i, s in enumerate(students.student_id)}; j_idx = {j: i for i, j in enumerate(jobs.job_id)}
fb["sidx"], fb["jidx"] = fb.student_id.map(s_idx), fb.job_id.map(j_idx)

# ---- 3. featurize ----
t = time.time()
fz = Featurizer().fit(students, jobs)
S = fz.encode_students(students.to_dict("records")); J = fz.encode_jobs(jobs.to_dict("records"))
S_rel = skill_relations(S, J)
print(f"{len(fz.skills)} skills | {len(fz.roles)} roles | dims={fz.dims} | featurized in {time.time()-t:.0f}s")
print("labelled pairs:", len(fb), "| label mix:", fb.y.value_counts(normalize=True).sort_index().round(2).to_dict())

In [ ]:
# ---- split (grouped by student so no student appears in two sets) ----
rng = np.random.default_rng(CFG["seed"])
if CFG["split"] == "time":
    order = fb.sort_values("recommendation_date").index.to_numpy(); n = len(order)
    te, va, tr = order[int(n*(1-CFG["test_frac"])):], order[int(n*(1-CFG["test_frac"]-CFG["val_frac"])):int(n*(1-CFG["test_frac"]))], order[:int(n*(1-CFG["test_frac"]-CFG["val_frac"]))]
else:
    stu = rng.permutation(fb.sidx.unique()); n = len(stu)
    te_s, va_s = set(stu[:int(n*CFG["test_frac"])]), set(stu[int(n*CFG["test_frac"]):int(n*(CFG["test_frac"]+CFG["val_frac"]))])
    tr = fb.index[~fb.sidx.isin(te_s | va_s)].to_numpy(); va = fb.index[fb.sidx.isin(va_s)].to_numpy(); te = fb.index[fb.sidx.isin(te_s)].to_numpy()
print(f"train {len(tr)} | val {len(va)} | test {len(te)} rows ({CFG['split']} split)")

S_dev = {k: v.to(DEVICE) for k, v in S.items()}; J_dev = {k: v.to(DEVICE) for k, v in J.items()}
R_S = torch.tensor(fb.sidx.to_numpy().copy(), device=DEVICE); R_J = torch.tensor(fb.jidx.to_numpy().copy(), device=DEVICE)
R_Y = torch.tensor(fb.y.to_numpy().copy(), device=DEVICE); R_ST = torch.tensor(fb.st.to_numpy().copy(), device=DEVICE, dtype=torch.long)
def get(idx):
    idx = torch.as_tensor(np.array(idx), device=DEVICE)
    return {k: v[R_S[idx]] for k, v in S_dev.items()}, {k: v[R_J[idx]] for k, v in J_dev.items()}, R_Y[idx], R_ST[idx]

## Train

In [ ]:
net = SkillStudioNet(fz.dims, S_rel, CFG["d"], CFG["h"], CFG["dropout"]).to(DEVICE)
print("parameters:", sum(p.numel() for p in net.parameters()))
bce = F.binary_cross_entropy_with_logits

def loss_fn(Sb, Jb, y, st):
    z, za, _ = net(Sb, Jb, paired=True)
    l1, l2 = net.rel_logits(z)
    loss = bce(l1, (y >= 1).float()) + bce(l2, (y >= 2).float())             # ordinal (cumulative) targets
    if CFG["aux_weight"] > 0:
        ok = st >= 0
        if ok.any():
            tgt = (st[ok, None] >= torch.arange(1, 5, device=DEVICE)).float()
            loss = loss + CFG["aux_weight"] * bce(net.status_logits(za)[ok], tgt)
    return loss

@torch.no_grad()
def val_loss(rows, bs=4096):
    net.eval(); tot = 0
    for a in range(0, len(rows), bs):
        Sb, Jb, y, st = get(rows[a:a+bs]); tot += loss_fn(Sb, Jb, y, st).item() * len(y)
    return tot / len(rows)

opt = torch.optim.AdamW(net.parameters(), lr=CFG["lr"], weight_decay=CFG["weight_decay"])
steps = CFG["epochs"] * int(np.ceil(len(tr) / CFG["batch_size"]))
sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=CFG["lr"], total_steps=steps, pct_start=0.15)
best, best_state, bad = 1e9, None, 0
for ep in range(CFG["epochs"]):
    net.train(); perm = rng.permutation(tr); run = 0
    for a in range(0, len(perm), CFG["batch_size"]):
        Sb, Jb, y, st = get(perm[a:a+CFG["batch_size"]])
        opt.zero_grad(); l = loss_fn(Sb, Jb, y, st); l.backward()
        torch.nn.utils.clip_grad_norm_(net.parameters(), 1.0); opt.step(); sched.step(); run += l.item() * len(y)
    vl = val_loss(va)
    print(f"epoch {ep+1:02d} | train {run/len(tr):.4f} | val {vl:.4f} | related-skill gate {torch.sigmoid(net.rel_gate).item():.2f}")
    if vl < best - 1e-4: best, bad, best_state = vl, 0, {k: v.detach().clone() for k, v in net.state_dict().items()}
    else:
        bad += 1
        if bad >= CFG["patience"]: print("early stop"); break
net.load_state_dict(best_state); net.eval()

In [ ]:
# ---- post-hoc calibration on the VALIDATION rows (3 scalars: slope + two thresholds; the rest of the network is frozen) ----
with torch.no_grad():
    zs, ys = [], []
    for a in range(0, len(va), 4096):
        Sb, Jb, y_, _ = get(va[a:a+4096]); z_, _, _ = net(Sb, Jb, paired=True); zs.append(z_); ys.append(y_)
    zv, yv = torch.cat(zs), torch.cat(ys)
cal_params = [net.rel_scale, net.rel_b2, net.rel_raw]
for p_ in net.parameters(): p_.requires_grad_(False)
for p_ in cal_params: p_.requires_grad_(True)
def cal_loss():
    l1, l2 = net.rel_logits(zv); return bce(l1, (yv >= 1).float()) + bce(l2, (yv >= 2).float())
print("validation loss before calibration:", round(cal_loss().item(), 4))
lb = torch.optim.LBFGS(cal_params, lr=0.5, max_iter=100)
def closure(): lb.zero_grad(); l = cal_loss(); l.backward(); return l
lb.step(closure)
print("validation loss after calibration: ", round(cal_loss().item(), 4), "| slope", round(net.rel_scale.item(), 3))
for p_ in net.parameters(): p_.requires_grad_(False)

## Evaluate (held-out test students)

In [ ]:
@torch.no_grad()
def predict_rows(rows, bs=4096):
    net.eval(); o = dict(match=[], probs=[], offer=[], cov=[])
    for a in range(0, len(rows), bs):
        Sb, Jb, _, _ = get(rows[a:a+bs]); z, za, pf = net(Sb, Jb, paired=True); m, pr = net.match_outputs(z)
        o["match"].append(m.cpu()); o["probs"].append(pr.cpu()); o["offer"].append(torch.sigmoid(net.status_logits(za)[:, 3]).cpu()); o["cov"].append(pf[:, F_IDX["req_cov_exact"]].cpu())
    return {k: torch.cat(v).numpy() for k, v in o.items()}

def pairwise_within_student(df, col):
    ok = tot = 0
    for _, g in df.groupby("sidx"):
        y, s = g.y.to_numpy(), g[col].to_numpy()
        for i in range(len(g)):
            for j in range(len(g)):
                if y[i] > y[j]: tot += 1; ok += (s[i] > s[j]) + 0.5 * (s[i] == s[j])
    return ok / max(tot, 1)

def to_class(score, c): return np.digitize(score, c)          # 0 Not / 1 Somewhat / 2 Relevant from two cut-points

# ordinal models rarely "argmax" the middle class, so tune two cut-points on the VALIDATION rows (never on test)
V = fb.loc[va].reset_index(drop=True); PV = predict_rows(va); best_f1, cuts = -1, (0.33, 0.66)
for c1 in np.arange(0.15, 0.6, 0.01):
    for c2 in np.arange(c1 + 0.05, 0.95, 0.01):
        f = f1_score(V.y, to_class(PV["match"], (c1, c2)), average="macro")
        if f > best_f1: best_f1, cuts = f, (c1, c2)
CFG["fit_cutoffs"] = [round(float(cuts[0]) * 100, 1), round(float(cuts[1]) * 100, 1)]
print("tuned cut-points on validation (match score %):", CFG["fit_cutoffs"], "| val macro-F1", round(best_f1, 3))

def report(name, score, y, df, probs=None):
    m = y != 1; r = dict(AUC_rel_vs_not=roc_auc_score(y[m] == 2, score[m]), spearman=spearmanr(score, y)[0], pairwise_in_student=pairwise_within_student(df.assign(s=score), "s"))
    if probs is not None:
        pred = to_class(score, cuts); r.update(accuracy=accuracy_score(y, pred), macro_F1=f1_score(y, pred, average="macro"), QWK=cohen_kappa_score(y, pred, weights="quadratic"))
    print(f"{name:34s}", {k: round(float(v), 3) for k, v in r.items()}); return {k: float(v) for k, v in r.items()}

T = fb.loc[te].reset_index(drop=True); P = predict_rows(te); y = T.y.to_numpy()
metrics = {"model": report("NEURAL MODEL", P["match"], y, T, P["probs"]),
           "baseline_coverage_only": report("baseline: required-skill coverage", P["cov"], y, T),
           "reference_old_match_score(LEAK)": report("reference: old match_score (leak)", T.match_score.to_numpy(), y, T)}
pr = P["probs"][:, 2]; bins = pd.cut(pr, [0, .1, .2, .3, .4, .5, .6, .7, .8, .9, 1.0], include_lowest=True)
cal = pd.DataFrame(dict(p=pr, obs=(y == 2), bin=bins)).groupby("bin", observed=True).agg(predicted=("p", "mean"), observed=("obs", "mean"), n=("p", "size")).round(3)
print("\nCalibration of P(Relevant):"); print(cal.to_string())
metrics["ECE_relevant"] = float((cal.n * (cal.predicted - cal.observed).abs()).sum() / cal.n.sum()); print("ECE:", round(metrics["ECE_relevant"], 4))
if (T.st >= 0).any():
    k = (T.st >= 0).to_numpy(); metrics["offer_AUC"] = float(roc_auc_score(T.st[k] == 4, P["offer"][k])); print("offer-likelihood AUC (aux head):", round(metrics["offer_AUC"], 3))
print("\nconfusion with tuned cut-points (rows=true, cols=pred):"); print(pd.crosstab(y, to_class(P["match"], cuts)).to_string())

## Save the .pt (with readiness calibration), reload it, demo

In [ ]:
net.cpu()
ckpt = build_checkpoint(net, fz, jobs, J, torch.zeros(1), CFG, metrics)
pred = SkillStudioPredictor(ckpt)                                   # uses GPU if available
sample = students.sample(min(CFG["readiness_ref_students"], len(students)), random_state=CFG["seed"])
tab = pred.calibrate_readiness(sample)                              # per-component p5-p95 scaling + cohort percentiles
ckpt.update(pred.export_calibration())
os.makedirs(CFG["out_dir"], exist_ok=True)
OUT = save_checkpoint(os.path.join(CFG["out_dir"], "skillstudio_model.pt"), ckpt)
print("saved", OUT, f"({os.path.getsize(OUT)/1e6:.1f} MB)")
print("readiness score over the sample [5,25,50,75,95]:", np.percentile(tab.readiness_score, [5, 25, 50, 75, 95]).round(1))
print("raw component ranges (p5, p95):", {k: [round(x, 4) for x in v] for k, v in pred.scale.items()})
print(tab[[k + "_scaled" for k in READINESS_KEYS]].quantile([.05, .5, .95]).round(2).to_string(), "\n(p5 ~ 0 and p95 ~ 1 for every component = scaling is healthy)")

In [ ]:
# ---- reload exactly as the app would, then compare readiness with observed outcomes ----
pred = SkillStudioPredictor.load(OUT)
test_students = students[students.student_id.isin(T.student_id.unique())]
rt = pred.readiness_table(test_students.head(1500))
obs = T.groupby("student_id").agg(mean_fb=("y", "mean"), offered=("st", lambda s: (s == 4).mean())).reset_index()
chk = rt.merge(obs, on="student_id")
print("Spearman(readiness, mean observed feedback):", round(spearmanr(chk.readiness_score, chk.mean_fb)[0], 3), "| vs offer rate:", round(spearmanr(chk.readiness_score, chk.offered)[0], 3))
print(chk.groupby(pd.qcut(chk.readiness_score, 5, duplicates="drop")).agg(students=("mean_fb", "size"), mean_feedback=("mean_fb", "mean"), offer_rate=("offered", "mean")).round(2).to_string())
print("(sanity check only: in this dataset outcomes depend on the shown PAIR, not on student-level quality, so a flat result is expected)")

In [ ]:
# ---- demo: what the app receives for one student ----
student = students.iloc[0].to_dict()                              # in the app: a dict with the same fields as students.csv
res = pred.analyze(student, top_n=5)                              # max 3 per role by default; pass max_per_role=None for pure ranking
print(json.dumps({k: res[k] for k in ("readiness", "skill_roi", "unrecognized_skills")}, indent=1, default=float))
for r in res["recommendations"]:
    print(f"\n{r['match_score']:5.1f}  {r['fit_label']:13s} {r['job_title']} @ {r['company']} ({r['work_mode']}, {r['location']}) | required met {r['required_skills_met']} | missing {r['missing_required']}")
print("\nmatch against specific jobs:", [(m['job_id'], m['match_score']) for m in pred.match_jobs(student, list(jobs.job_id[:3]))])

## OPTIONAL patch cell: recalibrate readiness inside an ALREADY-SAVED .pt (no retraining)
Use this if you trained with the earlier version and only need the readiness-scaling fix.
Run cells **1-4** (imports + model code) in a fresh kernel, then run this cell. It needs nothing from the training cells.

In [ ]:
PT_PATH = "/kaggle/working/outputs/skillstudio_model.pt"          # <-- your saved model
f_st = next(f for f in glob.glob(f"{CFG['data_dir']}/**/*.csv", recursive=True) if "student" in os.path.basename(f).lower() and "feedback" not in f.lower())
students = pd.read_csv(f_st)

ckpt = torch.load(PT_PATH, map_location="cpu", weights_only=True)
pred = SkillStudioPredictor(ckpt)
print("OLD scale:", {k: [round(x, 4) for x in v] for k, v in pred.scale.items()})
sample = students.sample(min(ckpt["cfg"].get("readiness_ref_students", 4000), len(students)), random_state=42)
tab = pred.calibrate_readiness(sample)
ckpt.update(pred.export_calibration())
save_checkpoint(PT_PATH, ckpt)
print("NEW scale:", {k: [round(x, 4) for x in v] for k, v in pred.scale.items()})
print("readiness score [5,25,50,75,95]:", np.percentile(tab.readiness_score, [5, 25, 50, 75, 95]).round(1))
print(tab[[k + "_scaled" for k in READINESS_KEYS]].quantile([.05, .5, .95]).round(2).to_string(), "\n(p5 ~ 0 and p95 ~ 1 for every component = scaling is healthy)")